In [25]:
import torch
from torch import nn

In [26]:
# Cell 2
class Module(nn.Module):
    """앞으로 만들 모델들이 공통으로 사용할 간단한 base class."""

    def __init__(self):
        # nn.Module이 parameter와 submodule을 관리할 수 있도록
        # 부모 class의 초기화 method를 먼저 실행한다.
        super().__init__()

    def forward(self, X):
        # 실제 신경망은 자식 class가 self.net에 저장한다고 가정한다.
        if not hasattr(self, "net"):
            raise AttributeError("Neural network self.net is not defined.")

        # 입력 X를 실제 신경망에 전달해 예측값을 반환한다.
        return self.net(X)

    def loss(self, predictions, labels):
        # 구체적인 loss 계산법은 자식 class가 정의해야 한다.
        raise NotImplementedError

    def training_step(self, batch):
        # 현재 batch는 (features, labels)로 구성되어 있다.
        features, labels = batch

        # model(features)를 호출하면
        # nn.Module.__call__을 거쳐 forward(features)가 실행된다.
        predictions = self(features)

        # 예측값과 실제 label로 scalar loss를 계산한다.
        loss = self.loss(predictions, labels)

        return loss

    def configure_optimizers(self):
        # 어떤 optimizer를 사용할지도 자식 class가 정의한다.
        raise NotImplementedError

In [27]:
# Cell 3
class LinearRegression(Module):
    def __init__(self, num_inputs, learning_rate=0.03):
        super().__init__()

        # learning rate는 optimizer를 만들 때 사용한다.
        self.learning_rate = learning_rate

        # num_inputs개의 feature를 받아 숫자 하나를 출력하는 선형층
        self.net = nn.Linear(
            in_features=num_inputs,
            out_features=1,
        )

    def loss(self, predictions, labels):
        # D2L의 제곱손실:
        # 각 sample의 1/2 * (prediction - label)^2를 평균낸다.
        errors = predictions - labels
        per_example_loss = 0.5 * errors.pow(2)
        return per_example_loss.mean()

    def configure_optimizers(self):
        # self.parameters()는 self.net의 weight와 bias를 반환한다.
        return torch.optim.SGD(
            self.parameters(),
            lr=self.learning_rate,
        )

In [28]:
features = torch.tensor([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
    [4.0, 3.0],
])

labels = torch.tensor([
    [1.0],
    [4.0],
    [-1.0],
    [2.0],
])

model = LinearRegression(
    num_inputs=2,
    learning_rate=0.03
)

predictions = model(features)

print("Predictions:")
print(predictions)
print("Shape:", predictions.shape)

Predictions:
tensor([[0.8709],
        [0.1689],
        [1.3460],
        [0.6441]], grad_fn=<AddmmBackward0>)
Shape: torch.Size([4, 1])


In [29]:
batch = (features, labels)
loss = model.training_step(batch)

print("Loss:", loss.item())
print("Loss shape:", loss.shape)

Loss: 2.754542350769043
Loss shape: torch.Size([])


In [30]:
optimizer = model.configure_optimizers()

optimizer.zero_grad()

loss.backward()

print("Weight gradient:")
print(model.net.weight.grad)

print("\nBias gradient:")
print(model.net.bias.grad)

Weight gradient:
tensor([[-1.5443,  0.3067]])

Bias gradient:
tensor([-0.7425])
